In [38]:
import os
import sys
import h5py
import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0,rf'{os.getcwd()}\Identification')

from scipy.ndimage import zoom
from scipy.signal import medfilt2d
from scipy.ndimage import gaussian_filter
from Helpers import sphereMask, saveData
from skimage.feature import peak_local_max

import torch
import torch.fft as fft


In [39]:
sphereData = rf'{os.getcwd()}\Data\sphereScans'

for scan in os.listdir(sphereData):




    ### Load Data ###--------------------------------------------------------------------

    # Load experiment
    with h5py.File(rf'{sphereData}\{scan}',"r") as f:
        print(list(f['RawData'].keys()))
        data = f['RawData'][scan.replace('.hdf5','')][()]

    # Note data size
    sz = data.shape

    # Transpose for xyz future access
    properData  = np.transpose(data,[1,2,0]) # -> (rows, width, slices)





    ### Data Processing ###-------------------------------------------------------------

    # Apply median filter for SP noise
    medData = np.zeros(np.shape(data))
    for i,img in enumerate(data, start=0):
        medData[i,:,:] = medfilt2d(img, kernel_size=3) #was 3 for soft particles/ 5 for the hard particle 

    # Normalize slice by slice
    goodBright = (np.mean(medData[:,:,0:sz[2]//4]) + np.mean(medData[:,:,3*sz[2]//4:]))/2
    sliceAvgs = np.mean(medData,axis=(0,2))

    brightData = np.zeros(sz) 
    for i in range(sz[1]):
        brightData[:,i,:] = medData[:,i,:]*goodBright/sliceAvgs[i]

    # Normalize total image
    normData = (brightData - np.min(brightData))/(np.max(brightData) - np.min(brightData))

    # Blur and brighten image
    std=3; strength=.2; revamp=15   # was 1.0 and 18 for hard particle/ was 0.2 and 15 for soft particles

    blurData = gaussian_filter(normData, sigma=std)
    sharpData = np.clip(revamp*(normData - blurData*strength), 0,1)





    ### Chunk Data ###-----------------------------------------------------------------

    # Display data dims
    r,c,z = data.shape
    print(f'Data dims: {r,c,z}')

    # Crop to proper data dims
    cropData = data[:447,:936,:944]; r,c,z = cropData.shape
    print(f'Data dims: {r,c,z}')

    if r % 2 == 1 and c % 4 == 0 and z % 4 == 0:
        print('Valid size')
    else:
        print('Requirements failed')
        raise TypeError

    # Create sphere conv mask
    kernel = sphereMask(diameter=126,pad=2,scale=.6)
    kP = kernel.shape[0]//2+1

    # Check for center point
    assert kernel.shape[0] % 2 == 1, f'Kernel must be odd: {kernel.shape}'
    assert kP % 2 == 1 ,             f'Kernel padding must be odd: kP={kP}'


    # Split data
    upper = cropData[:,:,   int(z/2)-kP: ] # U
    lower = cropData[:,:, 0:int(z/2)+kP  ] # L

    # Split into 4
    ul,uu = upper[:, :int(c/2)+kP ,:], upper[:, int(c/2)-kP: ,:]
    ll,lu = lower[:, :int(c/2)+kP ,:], lower[:, int(c/2)-kP: ,:]
    kChunks = [ll,ul,lu,uu]

    # Pad chunks
    pChunks = [np.pad(chunk,pad_width=kP) for chunk in kChunks]

    # Pad kernel to data chunk dimensions
    expansionPadding = [(chunkDim-kernelDim)//2 for chunkDim, kernelDim in zip(pChunks[0].shape, kernel.shape)]
    pKernel          = np.pad(kernel, pad_width=((expansionPadding[0],expansionPadding[0]),
                                                 (expansionPadding[1],expansionPadding[1]),
                                                 (expansionPadding[2],expansionPadding[2])))
    # Confirm sizes are equal
    print(f'Padded chunk size:  {pChunks[0].shape}')
    print(f'Padded kernel size: {pKernel.shape}')





    ### Convolve Data ###----------------------------------------------------------------

    # FFT kernel
    gKern = torch.from_numpy(pKernel).cuda()
    fgKern = fft.rfftn(fft.ifftshift(gKern))
    del gKern

    # Create convolutions of kernel and data chunks
    convMaps = []; chunkSz = pChunks[0].shape;
    for chunk in pChunks:

        # FFT data chunk
        gChunk = torch.from_numpy(chunk).cuda()
        fgChunk = fft.rfftn(gChunk)
        del gChunk

        # Convolve and IFFT
        fgChunk *= fgKern
        gChunk = fft.irfftn(fgChunk, s=chunkSz)
        del fgChunk

        # Gain back memory
        fChunk = gChunk.cpu()
        del gChunk

        # Add convolved chunk to list
        convMaps.append(fChunk)




    ### Recombine Data ###----------------------------------------------------------------

    def conjoin(inp):
        """Conjoin all chunks together into one matrix, and return result"""
    
        ll,ul,lu,uu = inp
        ll_unpad = ll[kP:-kP, kP:-kP, kP:-kP]  # Now shape (r, 266, z_chunk)
        ul_unpad = ul[kP:-kP, kP:-kP, kP:-kP]
        lu_unpad = lu[kP:-kP, kP:-kP, kP:-kP]  
        uu_unpad = uu[kP:-kP, kP:-kP, kP:-kP]

        # Combine column chunks
        columnComb1 = np.concatenate(
            (ll_unpad[:, :c//2, :],      # First 256 columns from ll
             lu_unpad[:, -c//2:, :]),    # Last 256 columns from lu
            axis=1
        )
        columnComb2 = np.concatenate(
            (ul_unpad[:, :c//2, :],      
             uu_unpad[:, -c//2:, :]),    
            axis=1
        )
        
        # Combine z chunks
        zedComb = np.concatenate(
            (columnComb1[:, :, :z//2],   
             columnComb2[:, :, -z//2:]), 
            axis=2
        )
    
        # Return unpadded, total array
        return zedComb[kP:-kP,kP:-kP,kP:-kP]

    # Reconjoin data
    conData = conjoin(convMaps)
    reData  = conjoin(pChunks)

    # Assure equivelence
    print(f'Unconvolved data: {cropData.shape}')
    print(f'Convolved data: {conData.shape}')
    print(f'Padded data:    {reData.shape}')







    ### Find data peaks ###--------------------------------------------------------------
    
    # Normalize conv data
    normConv = (conData - np.min(conData))/(np.max(conData) - np.min(conData))

    # Find peaks from conv
    peaks = peak_local_max(normConv,min_distance=30) 



    ### Save data ###--------------------------------------------------------------------

    saveDir  = rf'{os.getcwd()}\DataProcessed\sphereScans'
    scanName = scan.replace('.hdf5','')
    
    np.save(file=rf'{saveDir}\{scanName}.npy', arr=peaks)
    saveData(data=reData,  location=saveDir,saveName='raw'+scanName)
    saveData(data=normConv,location=saveDir,saveName='con'+scanName)

['Cycle100s1']
Data dims: (448, 1216, 1024)
Data dims: (447, 936, 944)
Valid size
Padded chunk size:  (529, 591, 595)
Padded kernel size: (529, 591, 595)
Unconvolved data: (447, 936, 944)
Convolved data: (365, 854, 862)
Padded data:    (365, 854, 862)
['Cycle100s2']
Data dims: (448, 1216, 1024)
Data dims: (447, 936, 944)
Valid size
Padded chunk size:  (529, 591, 595)
Padded kernel size: (529, 591, 595)
Unconvolved data: (447, 936, 944)
Convolved data: (365, 854, 862)
Padded data:    (365, 854, 862)
['Cycle100s3']
Data dims: (448, 1216, 1024)
Data dims: (447, 936, 944)
Valid size
Padded chunk size:  (529, 591, 595)
Padded kernel size: (529, 591, 595)
Unconvolved data: (447, 936, 944)
Convolved data: (365, 854, 862)
Padded data:    (365, 854, 862)
['Cycle100s4']
Data dims: (448, 1216, 1024)
Data dims: (447, 936, 944)
Valid size
Padded chunk size:  (529, 591, 595)
Padded kernel size: (529, 591, 595)
Unconvolved data: (447, 936, 944)
Convolved data: (365, 854, 862)
Padded data:    (365, 85

In [ ]:
print(scan)
print('con'+scan.replace('.hdf5',''))

Cycle100s1.hdf5
convCycle100s1
